# Protocols, capabilities, and the plain `Posterior`

Upper layers are written against small protocols rather than a model class:

* `SupportsPosterior` — anything with draws you can ask questions of.
* `SupportsIntervention` — anything you can ask a counterfactual of.
* `Capability` — what a producer can do; a request needing more returns a typed
  `Unsupported` rather than a wrong number.

`Posterior` is the sampler-free implementation of `SupportsPosterior`: a dict of arrays with
an npz round-trip. A NumPyro fit, a Laplace approximation, a file, and a hand-built dict are
all the same kind of thing to everything above `infer`.

In [ ]:
import numpy as np

from axiom.core import (
    Capability,
    D,
    Intervention,
    Posterior,
    PredictiveDraws,
    SupportsIntervention,
    SupportsPosterior,
    TimeWindow,
    Treatment,
    Unsupported,
    missing_capabilities,
)

## A hand-built posterior

Draws are `(chain, draw, *shape)`. `coords` label the trailing axes; `provenance` is
free-form metadata — always record the seed (the seed contract, note 0002.8).

In [ ]:
rng = np.random.default_rng(42)
post = Posterior(
    {
        "beta": rng.normal(0.3, 0.1, size=(4, 500, 2)),
        "sigma": rng.lognormal(-1, 0.2, size=(4, 500)),
    },
    coords={"treatment": ["fertilizer", "irrigation"]},
    provenance={"seed": 42, "backend": "hand-built"},
)
print(post)
print(isinstance(post, SupportsPosterior), post.names(), post.n_chains, post.n_draws())

In [ ]:
print(post.draws("beta").shape, "->", post.flat("beta").shape)
print(post.coords())
print(post.summary("sigma", definition="hdi", mass=0.9))

Every summary carries its interval definition and mass — there is no way to get a number out
of a `Posterior` without them.

In [ ]:
s = post.summary("sigma", definition="eti", mass=0.5)
print(s.interval.definition, s.interval.mass, s.interval)

## npz round-trip (no pickle)

`to_npz` writes the arrays plus a JSON metadata entry; `from_npz` loads with
`allow_pickle=False`. Gate 5 asserts no pickle anywhere in `axiom.io`.

In [ ]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
f = post.to_npz(tmp / "posterior.npz")
again = Posterior.from_npz(f)
print(again == post, again.provenance)
print(post.with_provenance(note="re-read").provenance)

## Interventions and capabilities

`SupportsIntervention` is the seam for counterfactuals. A producer declares its
`capabilities()`; before asking for something, check `missing_capabilities` and degrade with
a typed `Unsupported` instead of guessing.

In [ ]:
class ToyLinearSurface:
    """outcome = sum_k beta_k * dose_k, posterior over beta. Supports counterfactuals only."""

    def __init__(self, posterior: Posterior, treatments: list[Treatment]) -> None:
        self._post = posterior
        self._treatments = treatments

    @property
    def treatments(self) -> list[Treatment]:
        return self._treatments

    def capabilities(self) -> frozenset[Capability]:
        return frozenset({Capability.COUNTERFACTUAL})

    def predict_under(self, iv: Intervention, window: TimeWindow | None = None, seed: int | None = None) -> PredictiveDraws:
        beta = self._post.draws("beta")                       # (chain, draw, 2)
        dose = np.array([iv.doses.get(t.name, 0.0) for t in self._treatments])
        return PredictiveDraws(values=beta @ dose, intervention=iv, window=window, seed=seed)


surface = ToyLinearSurface(post, [Treatment(name="fertilizer", dimension=D.currency, unit="USD"),
                                  Treatment(name="irrigation", dimension=D.currency, unit="USD")])
print(isinstance(surface, SupportsIntervention))

In [ ]:
iv = Intervention(doses={"fertilizer": 100.0, "irrigation": 20.0}, version="granular-v2")
pred = surface.predict_under(iv, seed=0)
print(pred.values.shape, pred.intervention.treatments, pred.intervention.version)

In [ ]:
needed = {Capability.COUNTERFACTUAL, Capability.MARGINAL}
missing = missing_capabilities(surface, needed)
if missing:
    result = Unsupported(reason=f"surface lacks {', '.join(missing)}", missing=missing)
    print(result.status, "|", result.reason, "|", bool(result))

That `Unsupported` is what an estimand realization returns when the surface cannot support
it (gate 7, Phase 4). It is falsy, it carries a reason, and it serializes — it is never
`nan` and never an exception swallowed somewhere upstream.